In [2]:
import pandas as pd
import numpy as np
import os
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import joblib

csv_filename = 'verified_online.csv'

if not os.path.exists(csv_filename):
    print(f"🚨 ERROR: Cannot find '{csv_filename}'.")
else:
    print(f"Loading bad links from {csv_filename}...")
    
    # 1. LOAD THE BAD LINKS (And force them to be marked as Threats)
    bad_df = pd.read_csv(csv_filename)
    
    # We don't care if 'type' exists, we know these are all bad!
    # Just grab 5,000 of them so we don't crash your computer's memory
    bad_df = bad_df[['url']].dropna().head(5000)
    bad_df['is_malicious'] = 1 
    
    # 2. CREATE FAKE "SAFE" LINKS (So the AI learns what good looks like)
    print("Generating safe links to balance the AI's brain...")
    safe_domains = [
        "https://www.google.com", "https://www.youtube.com", "https://www.wikipedia.org", 
        "https://www.amazon.com", "https://www.apple.com", "https://www.microsoft.com",
        "https://github.com", "https://stackoverflow.com", "https://www.reddit.com",
        "https://www.netflix.com", "https://www.spotify.com", "https://www.linkedin.com"
    ]
    # Multiply the safe list so we have 5,000 safe links to match the 5,000 bad links
    safe_urls = safe_domains * 420 
    
    safe_df = pd.DataFrame({'url': safe_urls, 'is_malicious': 0})
    
    # 3. MASH THEM TOGETHER
    df = pd.concat([bad_df, safe_df], ignore_index=True)
    # Shuffle the deck so the AI doesn't memorize the order
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # ==========================================
    # 4. FEATURE EXTRACTION (INSTANT MATH)
    # ==========================================
    print(f"Extracting metrics from {len(df)} total URLs...")
    
    df['url_str'] = df['url'].astype(str)
    features_df = pd.DataFrame()
    
    features_df['url_length'] = df['url_str'].str.len()
    features_df['qty_dots'] = df['url_str'].str.count(r'\.')
    features_df['qty_hyphens'] = df['url_str'].str.count('-')
    features_df['qty_slash'] = df['url_str'].str.count('/')
    features_df['qty_at'] = df['url_str'].str.count('@')
    
    sensitive_words = 'login|secure|bank|verify|update'
    features_df['has_sensitive_word'] = df['url_str'].str.lower().str.contains(sensitive_words).astype(int)
    features_df['is_ip'] = df['url_str'].str.contains(r'(?:\d{1,3}\.){3}\d{1,3}').astype(int)
    
    X = features_df
    y = df['is_malicious']
    
    # ==========================================
    # 5. TRAIN THE AI & SAVE
    # ==========================================
    print("Training Machine Learning Model...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, eval_metric='logloss')
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions) * 100
    
    print(f"✅ Real Model Accuracy: {accuracy:.2f}%")
    
    joblib.dump(model, 'phishguard_model.pkl')
    print("✅ SUCCESS: 'phishguard_model.pkl' has been generated and saved!")

Loading bad links from verified_online.csv...
Generating safe links to balance the AI's brain...
Extracting metrics from 10040 total URLs...
Training Machine Learning Model...
✅ Real Model Accuracy: 99.60%
✅ SUCCESS: 'phishguard_model.pkl' has been generated and saved!
